# Bayesian PCA: choosing the number of components with ARD

PCA and PPCA fit the training data better and better as the latent dimension $L$ grows
([pcaOverfitDemo.ipynb](pcaOverfitDemo.ipynb)), so the likelihood cannot choose $L$.
Bayesian PCA (Bishop, 1999) puts an automatic relevance determination (ARD) prior on the columns of $\mathbf{W}$:

$$p(\mathbf{W} \mid \boldsymbol{\alpha}) = \prod_{l=1}^{L} \mathcal{N}(\mathbf{w}_l \mid \mathbf{0}, \alpha_l^{-1} \mathbf{I}).$$

Each column has its own precision $\alpha_l$, estimated from the data. Starting from more components than needed,
the $\alpha_l$ of the unneeded columns go to infinity, those columns shrink to zero, and the number of columns left
is the effective dimension. This notebook fits it with EM next to PPCA ([fa_em_demo.ipynb](fa_em_demo.ipynb)).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.stats import multivariate_normal

try:
    import probml_utils as pml
except ModuleNotFoundError:
    %pip install -qq git+https://github.com/probml/probml-utils.git
    import probml_utils as pml

## Data

$D = 10$ dimensions, $L = 3$ true latent directions with standard deviations 3, 2.5 and 2, isotropic noise
$\sigma^2 = 1$; 300 training and 300 test points.

In [ ]:
def make_data(N, D, L_true, sigma2=1.0, seed=0):
    rng = np.random.default_rng(seed)
    Q, _ = np.linalg.qr(rng.standard_normal((D, L_true)))
    W = Q * np.linspace(3, 2, L_true)  # orthogonal directions, std 3 ... 2
    X = rng.standard_normal((2 * N, L_true)) @ W.T + rng.standard_normal((2 * N, D)) * np.sqrt(sigma2)
    return X[:N], X[N:]


D, L_true = 10, 3
X, X_test = make_data(300, D, L_true)
N = len(X)

## EM for Bayesian PCA

The E step is the PPCA one, with $\mathbf{M} = \mathbf{W}^\top \mathbf{W} + \sigma^2 \mathbf{I}$:
$\mathbf{m}_n = \mathbf{M}^{-1} \mathbf{W}^\top \tilde{\mathbf{x}}_n$ and
$\mathbb{E}[\mathbf{z}_n \mathbf{z}_n^\top] = \sigma^2 \mathbf{M}^{-1} + \mathbf{m}_n \mathbf{m}_n^\top$.
The M step adds the prior to the PPCA update of $\mathbf{W}$, then re-estimates $\sigma^2$ and $\boldsymbol{\alpha}$:

$$\mathbf{W} = \left(\sum_n \tilde{\mathbf{x}}_n \mathbf{m}_n^\top\right) \left(\sum_n \mathbb{E}[\mathbf{z}_n \mathbf{z}_n^\top] + \sigma^2 \mathbf{A}\right)^{-1}, \quad \mathbf{A} = \mathrm{diag}(\boldsymbol{\alpha}),$$

$$\sigma^2 = \frac{1}{ND} \sum_n \left( \|\tilde{\mathbf{x}}_n\|^2 - 2 \mathbf{m}_n^\top \mathbf{W}^\top \tilde{\mathbf{x}}_n + \mathrm{tr}\left(\mathbb{E}[\mathbf{z}_n \mathbf{z}_n^\top] \mathbf{W}^\top \mathbf{W}\right) \right), \qquad \alpha_l = \frac{D}{\|\mathbf{w}_l\|^2}.$$

With $\boldsymbol{\alpha} = \mathbf{0}$ this is EM for PPCA, so one function fits both (`ard=False`).

In [ ]:
def loglik(X, mu, W, sigma2):
    return multivariate_normal(mu, W @ W.T + sigma2 * np.eye(len(mu))).logpdf(X).sum()


def ppca_em(X, L, ard=True, n_iter=5000, tol=1e-7, seed=0):
    N, D = X.shape
    mu = X.mean(axis=0)
    Xc = X - mu
    rng = np.random.default_rng(seed)
    W = rng.standard_normal((D, L))
    sigma2 = 1.0
    alpha = np.zeros(L)
    alphas, objective = [], []
    for it in range(n_iter):
        # E step
        Minv = np.linalg.inv(W.T @ W + sigma2 * np.eye(L))
        Mz = Xc @ W @ Minv  # row n is m_n, [N, L]
        Ezz = N * sigma2 * Minv + Mz.T @ Mz  # sum_n E[z_n z_n^T]
        # M step
        W = (Xc.T @ Mz) @ np.linalg.inv(Ezz + sigma2 * np.diag(alpha))
        sigma2 = (np.sum(Xc**2) - 2 * np.sum(Mz * (Xc @ W)) + np.trace(Ezz @ W.T @ W)) / (N * D)
        if ard:
            alpha = D / np.maximum(np.sum(W**2, axis=0), 1e-12)  # a switched-off column has alpha -> infinity
            alphas.append(alpha)
        # EM increases log p(X | W, sigma^2) + log p(W | alpha); stop when it settles. The log-likelihood alone
        # is not enough: it can level off while a column is still collapsing, and the log prior is not
        lp = loglik(X, mu, W, sigma2)
        if ard:
            lp += np.sum(0.5 * D * np.log(alpha / (2 * np.pi)) - 0.5 * alpha * np.sum(W**2, axis=0))
        objective.append(lp)
        if it > 0 and abs(objective[-1] - objective[-2]) < tol * abs(objective[-1]):
            break
    return mu, W, sigma2, np.array(alphas), np.array(objective)


L_max = D - 1
mu, W_bpca, sigma2_bpca, alphas, obj = ppca_em(X, L_max)
_, W_ppca, sigma2_ppca, _, obj_ppca = ppca_em(X, L_max, ard=False)
for name, o in [("Bayesian PCA", obj), ("PPCA", obj_ppca)]:
    assert np.all(np.diff(o) > -1e-6 * np.abs(o[1:])), f"{name}: EM objective decreased"

col_var = np.sum(W_bpca**2, axis=0)  # ||w_l||^2 = D / alpha_l
order = np.argsort(-col_var)
active = col_var > 1e-3 * col_var.max()
L_eff = active.sum()
print(f"Bayesian PCA: {len(obj)} iterations, effective L = {L_eff} (true L = {L_true}), sigma^2 = {sigma2_bpca:.3f}")
print("  ||w_l||^2, largest first:", np.array2string(col_var[order], precision=3, suppress_small=True))
print(f"PPCA with L = {L_max}: {len(obj_ppca)} iterations, sigma^2 = {sigma2_ppca:.3f}")
print("  ||w_l||^2:", np.array2string(np.sort(np.sum(W_ppca**2, axis=0))[::-1], precision=3))

## The columns of $\mathbf{W}$

A Hinton-style view of $|\mathbf{W}|$ ($D$ rows, $L = 9$ columns, sorted by $\|\mathbf{w}_l\|$). PPCA uses all nine
columns, fitting noise with the extra six; Bayesian PCA keeps three and sets the rest to zero.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4.5))
vmax = np.abs(W_ppca).max()
for ax, W, name in [(axs[0], W_ppca, "PPCA"), (axs[1], W_bpca, "Bayesian PCA")]:
    W = W[:, np.argsort(-np.sum(W**2, axis=0))]
    im = ax.imshow(np.abs(W), cmap="Blues", vmin=0, vmax=vmax)
    ax.set_title(f"{name}: $|\\mathbf{{W}}|$")
    ax.set_xticks(range(L_max))
    ax.set_xticklabels(range(1, L_max + 1))
    ax.set_yticks([])
    ax.set_xlabel("column $l$")
fig.colorbar(im, ax=axs, shrink=0.8)
pml.savefig("bayesianPcaHinton.pdf")
plt.show()

## Columns switching off

$1/\alpha_l = \|\mathbf{w}_l\|^2 / D$ per EM iteration: the columns Bayesian PCA keeps level off, the others fall
towards zero (the flat ends at about $10^{-13}$ are the floor `ppca_em` puts on $\|\mathbf{w}_l\|^2$ to keep
$\alpha_l$ finite).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for l in range(L_max):
    kept = active[l]
    ax.plot(
        np.arange(1, len(alphas) + 1), 1 / alphas[:, l],
        color="#2a78d6" if kept else "0.6", lw=2 if kept else 1,
        label=("kept" if kept else "switched off") if l in (np.argmax(active), np.argmin(active)) else None,
    )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel(r"$1 / \alpha_l$")
ax.legend()
fig.tight_layout()
pml.savefig("bayesianPcaAlpha.pdf")
plt.show()

## Compared with the PPCA likelihood

PPCA fitted for each $L$ (closed-form MLE): the training log-likelihood keeps rising with $L$, while the test
log-likelihood stops improving at the true dimension. Bayesian PCA picks $L$ from the training data alone.

In [ ]:
def ppca_mle(X, L):
    mu = X.mean(axis=0)
    evals, evecs = np.linalg.eigh(np.cov(X.T, bias=True))
    evals, evecs = evals[::-1], evecs[:, ::-1]
    sigma2 = evals[L:].mean()
    return mu, evecs[:, :L] * np.sqrt(evals[:L] - sigma2), sigma2


Ls = np.arange(1, D)
ll_train, ll_test = [], []
for L in Ls:
    mu_L, W_L, s2_L = ppca_mle(X, L)
    ll_train.append(loglik(X, mu_L, W_L, s2_L) / N)
    ll_test.append(loglik(X_test, mu_L, W_L, s2_L) / len(X_test))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(Ls, ll_train, "o-", color="#2a78d6", lw=2, label="PPCA, training")
ax.plot(Ls, ll_test, "o-", color="#eb6834", lw=2, label="PPCA, test")
ax.axvline(L_eff, color="0.4", ls="--", lw=1, label=f"Bayesian PCA, L = {L_eff}")
ax.set_xlabel("L")
ax.set_ylabel("log-likelihood per point")
ax.legend()
fig.tight_layout()
pml.savefig("bayesianPcaLoglik.pdf")
plt.show()
print("best test L:", Ls[np.argmax(ll_test)])

## Other true dimensions

The same fit, starting from $L = D - 1$ each time, on data with 1 to 6 true latent directions.

In [ ]:
for L_true_k in range(1, 7):
    X_k, _ = make_data(300, D, L_true_k, seed=L_true_k)
    _, W_k, _, _, _ = ppca_em(X_k, L_max)
    v = np.sum(W_k**2, axis=0)
    print(f"true L = {L_true_k}: Bayesian PCA keeps {np.sum(v > 1e-3 * v.max())}")

Bayesian PCA finds the true dimension from the training data alone: the prior makes each extra column pay for
itself, so columns that would only fit noise are switched off. PPCA uses every column it is given, and choosing
$L$ for it needs held-out data or a criterion such as BIC.

The prior here is ARD with point estimates of $\boldsymbol{\alpha}$ (type-II maximum likelihood); a fully Bayesian
treatment puts gamma priors on $\boldsymbol{\alpha}$ and $\sigma^{-2}$ and uses variational inference (Bishop, 1999b).